# RF impedance matching
Illustrative series RLC circuit, not measured antenna data. Run the code cell with Shift + Enter.
Requirements: numpy, pandas, matplotlib.
Reference impedance: 50 ohms. Resonance: 2.4 GHz. Resistance: 60 ohms.
S11 is 20 log10(|Gamma|); mismatch loss describes reflected power only, not radiation efficiency.
The notebook writes a CSV and PNG in its working folder. Exact zero reflection is clipped to -300 dB for plotting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

frequency_hz = np.linspace(1e9, 3e9, 1001)
z0 = 50.0  # ohms
resistance = 60.0  # ohms
inductance = 5e-9  # H
f0 = 2.4e9  # Hz
capacitance = 1 / ((2 * np.pi * f0)**2 * inductance)
omega = 2 * np.pi * frequency_hz
z_load = resistance + 1j * (omega * inductance - 1 / (omega * capacitance))
gamma = (z_load - z0) / (z_load + z0)
gamma_mag = np.abs(gamma)
s11_db = 20 * np.log10(np.maximum(gamma_mag, 1e-15))
vswr = (1 + gamma_mag) / (1 - gamma_mag)
mismatch_loss_db = -10 * np.log10(1 - gamma_mag**2)
assert np.all(gamma_mag < 1)
assert np.all(vswr >= 1)
assert np.all(mismatch_loss_db >= 0)
best = np.argmin(s11_db)
assert np.isclose(frequency_hz[best], f0)
assert np.isclose(vswr[best], 1.2)
results = pd.DataFrame({
    "frequency_GHz": frequency_hz / 1e9,
    "resistance_ohm": z_load.real,
    "reactance_ohm": z_load.imag,
    "S11_dB": s11_db,
    "VSWR": vswr,
    "mismatch_loss_dB": mismatch_loss_db,
})
results.to_csv("impedance_matching_results.csv", index=False)
print(f"Minimum S11 at {frequency_hz[best]/1e9:.3f} GHz: {s11_db[best]:.2f} dB")
print(f"VSWR: {vswr[best]:.2f}")
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(frequency_hz / 1e9, s11_db)
axes[0].axhline(-10, color="red", linestyle="--", label="-10 dB")
axes[0].set_ylabel("S11 (dB)")
axes[0].set_title("Illustrative series RLC load")
axes[0].legend()
axes[0].grid(True)
axes[1].plot(frequency_hz / 1e9, vswr)
axes[1].set_xlabel("Frequency (GHz)")
axes[1].set_ylabel("VSWR")
axes[1].grid(True)
plt.tight_layout()
plt.savefig("impedance_matching.png", dpi=150)
plt.show()
